# IPUMS [NHGIS](https://www.nhgis.org) Data Extraction Using [ipumsr](https://cran.r-project.org/web/packages/ipumsr/index.html) - Supplemental Exercise 2
### by [Kate Vavra-Musser](https://vavramusser.github.io) for the [R Spatial Notebook Series](https://vavramusser.github.io/r-spatial)

## Introduction
This notebook provides an additional example of the IPUMS NHGIS data extraction process using the IPUMS API via the ipumsr R package.  This exercise is a supplement to the workflow introducted in Chapter 3.4 IPUMS NHGIS Data Extraction Using ipumsr.

### Notebook Goals
This notebook replicates the IPUMS NHGIS data extraction process and extracts a NHGIS polygon shapefile.  The resulting data file is used in subsequent notebooks in the R Spatial Notebooks series.  The notebook provides an example of extracting spatial data only (without associated attribute data) from the IPUMS NHGIS repository.

### ✨ Prerequisites ✨
* Complete [Introduction to IPUMS and the IPUMS API](https://platform.i-guide.io/notebooks/82d3b176-e4e6-4307-8186-318a3fe6c81a)
* Set Up Your [IPUMS Account and API Key](https://account.ipums.org/api_keys)
* Complete [Introduction to sf: Reading, Writing, and Inspecting Vector Data](https://platform.i-guide.io/notebooks/9968babe-22e4-4c3d-98e2-d8b45e9672cd)
* Complete [IPUMS NHGIS Data Extraction Using ipumsr](https://platform.i-guide.io/notebooks/be08e56e-1c08-458e-a230-263c64d386bc)

### Notebook Overview
1. Setup
2. Extraction Workflow: Shapefiles Only

---

## 1. Setup
This section will guide you through the process of installing essential packages and setting your IPUMS API key.

#### Required Packages

[**dplyr**](https://cran.r-project.org/web/packages/dplyr/index.html) A Grammar of Data Manipulation. This notebook uses the the following functions from *dplyr*.

* [*filter*](https://rdrr.io/cran/dplyr/man/filter.html) · keep rows that match a condition

[**ipumsr**](https://cran.r-project.org/web/packages/ipumsr/index.html) An R Interface for Downloading, Reading, and Handling IPUMS Data.  This notebook uses the the following functions from *ipumsr*.

* [*define_extract_nhgis*](https://rdrr.io/cran/ipumsr/man/define_extract_nhgis.html) · define an IPUMS NHGIS extract request
* [*download_extract*](https://rdrr.io/cran/ipumsr/man/download_extract.html) · download a completed IPUMS data extract
* [*get_metadata_nhgis*](https://rdrr.io/cran/ipumsr/man/get_metadata_nhgis.html) · list available data sources from IPUMS NHGIS
* [*read_ipums_sf*](https://rdrr.io/cran/ipumsr/man/read_ipums_sf.html) · read spatial data from an IPUMS extract
* [*set_ipums_api_key*](https://rdrr.io/cran/ipumsr/man/set_ipums_api_key.html) · set your IPUMS API key
* [*submit_extract*](https://rdrr.io/cran/ipumsr/man/submit_extract.html) · submit an extract request via the IPUMS API
* *tst_spec* · create a *tst_spec* object containing a time series table specification
* [*wait_for_extract*](https://rdrr.io/cran/ipumsr/man/wait_for_extract.html) · wait for an extract to finish processing

[**sf**](https://cran.r-project.org/web/packages/sf/index.html) Support for simple features, a standardized way to encode spatial vector data. Binds to 'GDAL' for reading and writing data, to 'GEOS' for geometrical operations, and to 'PROJ' for projection conversions and datum transformations. Uses by default the 's2' package for spherical geometry operations on ellipsoidal (long/lat) coordinates.  This notebook uses the following functions from *sf*.

* [*st_write*](https://rdrr.io/cran/sf/man/st_write.html) · Write simple features object to file or database

### 1a. Install and Load Required Packages
If you have not already installed the required packages, uncomment and run the code below:

In [ ]:
# install.packages(c("dplyr", "ipumsr", "purr", "sf"))  

Load the packages into your workspace.

In [2]:
library(dplyr)
library(ipumsr)
library(purrr)
library(sf)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Linking to GEOS 3.10.2, GDAL 3.4.1, PROJ 7.2.1; sf_use_s2() is TRUE



### 1b. Set Your IPUMS API Key

Store your [IPUMS API key](https://account.ipums.org/api_keys) in your environment using the following code.

Refer to [Chapter 1.1 Introduction to IPUMS and the IPUMS API](https://platform.i-guide.io/notebooks/82d3b176-e4e6-4307-8186-318a3fe6c81a) for instructions on setting up your IPUMS account and API key.

In [3]:
ipumps_api_key = readline("Please enter your IPUMS API key: ")
set_ipums_api_key(ipumps_api_key, save = T, overwrite = T)

Existing .Renviron file copied to C:\Users\vavra\Documents/.Renviron_backup for backup purposes.

The environment variable IPUMS_API_KEY has been set and saved for future sessions.



## 2. NHGIS Polygons Only

### 2a. View and Filter the List of Geography Shapefiles

Forthis exercise, we only want the geography shapefile and aren't interested in downloading any data from the time-series tables NHGIS repository.  Therefore, we will jump directly into e will taking a look at the list of geography shapefiles that fit our critera.  We are looking for state boundary shapefiles so let's filter the shapefile metadata to only incluede shapefiles which include the word "state" on the description of their *geographic_level*.  We will also focus on only shapefiles using the 2010 Tiger-Line files so we will also filter based on the *year = 2010* criteria.

In [4]:
metadata_shp <- get_metadata_nhgis("shapefiles") %>%
    filter(year == 2010 & grepl("state", geographic_level, ignore.case = T)) %>%
    print(n = Inf)

Warning message:
"`get_metadata_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `get_metadata_catalog()` to obtain summary metadata or
  `get_metadata()` to obtain detailed metadata."


# A tibble: 7 x 6
  name                            year  geographic_level   extent basis sequence
  <chr>                           <chr> <chr>              <chr>  <chr>    <int>
1 us_state_2010_tl2010            2010  State              Unite~ 2010~      597
2 us_state_cenpop_2010_cenpop2010 2010  State (Centers of~ Unite~ 2010~      598
3 us_state_2010_tl2020            2010  State              Unite~ 2020~      599
4 us_stleg_up_2010_tl2010         2010  State Legislative~ Unite~ 2010~      618
5 us_stleg_up_2010_tl2020         2010  State Legislative~ Unite~ 2020~      619
6 us_stleg_lo_2010_tl2010         2010  State Legislative~ Unite~ 2010~      620
7 us_stleg_lo_2010_tl2020         2010  State Legislative~ Unite~ 2020~      621


This filter resulted in a list of sevem potential shapefiles.  Let's select the 2010 shapefile based on 2010 Tiger-Line shapefiles for states (*us_state_2010_tl2010*).

### 2b. Shapefile Extraction Specification and Submission

Now that we have selected our shapefile (*us_state_2010_tl2010*) we are ready to define and submit our extraction request to the IPUMS API.

In [5]:
extract_definition <- define_extract_nhgis(description = "I-GUIDE NHGIS State Polygons Shapefile Extraction",
                                           shapefiles = "us_state_2010_tl2010")

Warning message:
"`define_extract_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `define_extract_agg()` instead."


Submitting the extraction definition object *extract_definition* to the API.

In [6]:
extraction_submitted <- submit_extract(extract_definition)
extraction_complete <- wait_for_extract(extraction_submitted)
extraction_complete$status
filepath <- download_extract(extraction_submitted, overwrite = T)

Successfully submitted IPUMS NHGIS extract number 123

Checking extract status...

Waiting 10 seconds...

Checking extract status...

IPUMS NHGIS extract 123 is ready to download.



[1] "completed"

  |======================================================================| 100%


Shapefile saved to C:/Users/vavra/Dropbox/R Spatial/r-spatial/03 IPUMS Data Acquisition and Extraction/nhgis0123_shape.zip



The result of the extraction request will be one file, the state boundaries geography shapefile.  The next step is to read that file into R.

In [7]:
shp <- read_ipums_sf(filepath)

Let's take a look at the dimesions of the shapefile (*shp*).

In [8]:
dim(shp)

[1] 52 18

The shapefile includes 18 variables for 52 polygons inckding the 50 states, Washington D.C., and Puerto Rico.

Let's take a look at the first few lines of the shapefile

In [9]:
head(shp)

ERROR while rich displaying an object: Error in loadNamespace(x): there is no package called 'geojsonio'

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_geojson.sf(obj)
8. repr_geojson(geo

REGION10,DIVISION10,STATEFP10,STATENS10,GEOID10,STUSPS10,NAME10,LSAD10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,GISJOIN,Shape_area,Shape_len,geometry
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<MULTIPOLYGON [m]>
4,8,56,01779807,56,WY,Wyoming,00,G4000,A,251470069067,1864445306,+42.9918024,-107.5419255,G560,253333624652,2029324.3,MULTIPOLYGON (((-633753.4 8...
1,2,42,01779798,42,PA,Pennsylvania,00,G4000,A,115883064314,3397122731,+40.9042486,-077.8280624,G420,117307345229,1683454.2,MULTIPOLYGON (((1743857 454...
2,3,39,01085497,39,OH,Ohio,00,G4000,A,105828706692,10269012119,+40.4149297,-082.7119975,G390,106865663467,1920794.1,MULTIPOLYGON (((1073991 515...
4,8,35,00897535,35,NM,New Mexico,00,G4000,A,314160748240,756659673,+34.4391265,-106.1261511,G350,314918019164,2392830.3,MULTIPOLYGON (((-616758.6 -...
3,5,24,01714934,24,MD,Maryland,00,G4000,A,25141638381,6989579585,+38.9466584,-076.6744939,G240,25577994798,8688178.9,MULTIPOLYGON (((1738939 231...
1,1,44,01219835,44,RI,Rhode Island,00,G4000,A,2677566454,1323668539,+41.5978358,-071.5252895,G440,2817469209,841958.3,MULTIPOLYGON (((2009795 677...


Finally, we will save the shapefile.  Before we do that however, we subset out data to only the column we will use in subsequent analyses.  This will make it easier to save and work work this data.

In [10]:
colnames(shp)

[1] "REGION10"   "DIVISION10" "STATEFP10"  "STATENS10"  "GEOID10"   
 [6] "STUSPS10"   "NAME10"     "LSAD10"     "MTFCC10"    "FUNCSTAT10"
[11] "ALAND10"    "AWATER10"   "INTPTLAT10" "INTPTLON10" "GISJOIN"   
[16] "Shape_area" "Shape_len"  "geometry"

In [11]:
shp_cols <- c("STATEFP10", "STUSPS10", "NAME10", "REGION10")
shp <- shp[shp_cols]

We are ready to save the shapefile to our workspace.

In [14]:
st_write(shp, "ipums_nhgis_states", driver = "ESRI Shapefile", delete_dsn = T)

Deleting source `ipums_nhgis_states' using driver `ESRI Shapefile'
Writing layer `ipums_nhgis_states' to data source 
  `ipums_nhgis_states' using driver `ESRI Shapefile'
Writing 52 features with 4 fields and geometry type Multi Polygon.


At the end of this notebook we have saved a copy of the geographic data file for US states to the shapefile *ipums_nhgis_states.shp*.